In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
N = 6000

categories = ["Apparel", "Electronics", "Home", "Footwear", "Beauty"]
cat_probs = [0.32, 0.22, 0.18, 0.18, 0.10]
payment_methods = ["COD", "Prepaid_Card", "Prepaid_UPI", "Wallet"]
pay_probs = [0.42, 0.24, 0.24, 0.10]

product_category = rng.choice(categories, size=N, p=cat_probs)
payment_method = rng.choice(payment_methods, size=N, p=pay_probs)

base_price = {
    "Apparel": (400, 2200), "Electronics": (1200, 45000), "Home": (300, 8000),
    "Footwear": (500, 4500), "Beauty": (150, 2500),
}
price_inr = np.round(np.array([rng.uniform(*base_price[c]) for c in product_category]), 0)

discount_pct = np.clip(rng.normal(22, 15, N), 0, 75)
customer_tenure_days = np.clip(rng.exponential(380, N), 1, 2500).round(0)
num_previous_orders = np.clip((customer_tenure_days / 45) + rng.normal(0, 2, N), 0, None).round(0)
base_return_rate = np.clip(rng.beta(1.5, 9, N), 0, 1)
num_previous_returns = np.round(base_return_rate * num_previous_orders).clip(0, num_previous_orders)

delivery_distance_km = np.clip(rng.gamma(3, 90, N), 2, 2200).round(1)
delivery_days = np.clip(rng.normal(4.5, 2.2, N), 1, 21).round(0)
is_weekend_order = rng.integers(0, 2, N)

rating_given = rng.integers(1, 6, N).astype(float)
missing_mask = rng.random(N) < np.where(payment_method == "COD", 0.22, 0.06)
rating_given[missing_mask] = np.nan

fit_risk_cat = np.isin(product_category, ["Apparel", "Footwear"]).astype(float)
prev_return_ratio = np.where(num_previous_orders > 0,
                              num_previous_returns / np.maximum(num_previous_orders, 1), 0)

z = (-2.2 + 1.9 * prev_return_ratio + 0.55 * fit_risk_cat
     + 0.014 * (discount_pct - 20) / 10 + 0.9 * (payment_method == "COD").astype(float)
     + 0.10 * (delivery_days - 4.5) / 2 + 0.30 * (price_inr / base_price["Electronics"][1])
     + 0.05 * is_weekend_order - 0.15 * np.tanh(customer_tenure_days / 500))
prob_return = 1 / (1 + np.exp(-z))
returned = (rng.random(N) < prob_return).astype(int)

df = pd.DataFrame({
    "order_id": np.arange(1, N + 1), "product_category": product_category,
    "price_inr": price_inr, "discount_pct": np.round(discount_pct, 1),
    "payment_method": payment_method, "customer_tenure_days": customer_tenure_days.astype(int),
    "num_previous_orders": num_previous_orders.astype(int),
    "num_previous_returns": num_previous_returns.astype(int),
    "delivery_distance_km": delivery_distance_km, "delivery_days": delivery_days.astype(int),
    "is_weekend_order": is_weekend_order, "rating_given": rating_given, "returned": returned,
})
df.to_csv("orders_dataset.csv", index=False)
print("Rows:", len(df), "| Return rate:", round(df["returned"].mean(), 4))


Rows: 6000 | Return rate: 0.2275


EXPLARATORY DATA ANALYSIS

In [2]:
print(df.columns)

Index(['order_id', 'product_category', 'price_inr', 'discount_pct',
       'payment_method', 'customer_tenure_days', 'num_previous_orders',
       'num_previous_returns', 'delivery_distance_km', 'delivery_days',
       'is_weekend_order', 'rating_given', 'returned'],
      dtype='object')


In [3]:
print('Total number of rows:',len(df))
print('Overall returned rate: ', df['returned'].mean())
print('Percentage of missing rating: ',df['rating_given'].isna().mean()*100)

print("\nReturn rate by category:")
print(df.groupby("product_category")["returned"].mean())

print("\nReturn rate by payment method:")
print(df.groupby("payment_method")["returned"].mean())

print("\nMissing rating_given rate by payment method:")
print(df.groupby("payment_method")["rating_given"].apply(lambda x: x.isna().mean()))



Total number of rows: 6000
Overall returned rate:  0.2275
Percentage of missing rating:  13.05

Return rate by category:
product_category
Apparel        0.264275
Beauty         0.200345
Electronics    0.186930
Footwear       0.259570
Home           0.191469
Name: returned, dtype: float64

Return rate by payment method:
payment_method
COD             0.307477
Prepaid_Card    0.168154
Prepaid_UPI     0.169199
Wallet          0.178451
Name: returned, dtype: float64

Missing rating_given rate by payment method:
payment_method
COD             0.228309
Prepaid_Card    0.063143
Prepaid_UPI     0.056630
Wallet          0.063973
Name: rating_given, dtype: float64


**"The missingness in rating_given is MAR, conditional on payment_method: COD orders are missing ratings at ~22%, versus ~6% for all other payment methods — a clear dependency on an observed column, ruling out MCAR. It is not MNAR because the missingness is explained by payment method, not by the unobserved rating value itself.**

In [4]:
#DATA PRE PROCESSING 
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X=df.drop(columns=['returned','order_id'])
y=df['returned']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)


numeric_columns_features=["price_inr", "discount_pct", "customer_tenure_days",
                     "num_previous_orders", "num_previous_returns",
                     "delivery_distance_km", "delivery_days",
                     "is_weekend_order", "rating_given"]
categoricalfeatures=['product_category','payment_method']

numeric_pipeline= Pipeline([
    ("impute",SimpleImputer(strategy='median')),
    ("scale",StandardScaler())
])

categorical_pipeline= Pipeline([
    ('impute',SimpleImputer(strategy="most_frequent")),
    ("encode",OneHotEncoder(handle_unknown="ignore"))
])

preprocessor=ColumnTransformer([
    ("num",numeric_pipeline,numeric_columns_features),
    ("cat",categorical_pipeline,categoricalfeatures)
])

X_train_processed=preprocessor.fit_transform(X_train)
X_test_processed=preprocessor.transform(X_test)


print("Training shape: ",X_train_processed.shape)
print('Testing shape: ',X_test_processed.shape)

Training shape:  (4800, 18)
Testing shape:  (1200, 18)


In [5]:
#dumb baseline model 
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score,f1_score

dummy=DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_processed,y_train)
dummy_predictions=dummy.predict(X_test_processed)

print("Dummy model Accuracy: ", accuracy_score(y_test,dummy_predictions))
print("Dummy F1 Score(returned=1): ",f1_score(y_test,dummy_predictions,pos_label=1))

Dummy model Accuracy:  0.7725
Dummy F1 Score(returned=1):  0.0


*The dummy baseline achieves 77.25% accuracy simply by always predicting "not returned," since that's the majority class. But its F1-score for the returned=1 class is 0.0 — it never correctly identifies a single actual return. This shows the "high accuracy, zero recall" trap: on imbalanced data, accuracy alone can look good while the model is completely blind to the outcome that actually matters for the business.*

In [6]:
#training a logistic regression model 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score,recall_score,precision_score,roc_auc_score

logreg=LogisticRegression(class_weight='balanced',max_iter=1000,random_state=42)
logreg.fit(X_train_processed,y_train)

predictions=logreg.predict(X_test_processed)
probability=logreg.predict_proba(X_test_processed)[:,1]

print('Accuracy: ',accuracy_score(y_test,predictions))
print('F1 Score: ', f1_score(y_test,predictions,pos_label=1))
print("Recall (returned=1):", recall_score(y_test, predictions, pos_label=1))
print("Precision (returned=1):", precision_score(y_test, predictions, pos_label=1))
print("ROC-AUC:", roc_auc_score(y_test, probability))

Accuracy:  0.5916666666666667
F1 Score:  0.3920595533498759
Recall (returned=1): 0.5787545787545788
Precision (returned=1): 0.2964352720450281
ROC-AUC: 0.6252632660399652


In [7]:
thresholds = np.arange(0.1, 0.91, 0.02)
results = []

for t in thresholds:
    preds_t = (probability >= t).astype(int)
    f1 = f1_score(y_test, preds_t, pos_label=1)
    recall = recall_score(y_test, preds_t, pos_label=1)
    precision = precision_score(y_test, preds_t, pos_label=1)
    results.append((t, f1, recall, precision))

results_df = pd.DataFrame(results, columns=["threshold", "f1", "recall", "precision"])

# Find the threshold with the best F1
best_row = results_df.loc[results_df["f1"].idxmax()]
print("Best threshold:", best_row["threshold"])
print("F1 at best threshold:", best_row["f1"])
print("Recall at best threshold:", best_row["recall"])
print("Precision at best threshold:", best_row["precision"])

Best threshold: 0.44000000000000006
F1 at best threshold: 0.4090909090909091
Recall at best threshold: 0.7582417582417582
Precision at best threshold: 0.28010825439783493


/Users/harinisaravanan05/Desktop/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


**Lowering the decision threshold from 0.50 to 0.44 increases recall from 58% to 76% — a jump of about 18 percentage points — meaning the model now catches far more real returns before they happen. The cost is a small drop in precision (30% → 28%), meaning slightly more false alarms (orders flagged as risky that turn out fine). This trade-off makes sense if missing a real return is more costly to the business than double-checking a handful of orders that turn out to be safe.**

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score

rf = RandomForestClassifier(class_weight="balanced", random_state=42)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [6, 10, None]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(rf, param_grid, scoring="roc_auc", cv=cv, n_jobs=-1)
grid_search.fit(X_train_processed, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)

# Evaluate the best model on the held-out test set
best_rf = grid_search.best_estimator_
test_probs = best_rf.predict_proba(X_test_processed)[:, 1]
print("Test-set ROC-AUC:", roc_auc_score(y_test, test_probs))

Best params: {'max_depth': 6, 'n_estimators': 100}
Best CV ROC-AUC: 0.6179484601771899
Test-set ROC-AUC: 0.6143098181933133


In [9]:
import pandas as pd
from sklearn.inspection import permutation_importance

# Get feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()

# Method 1: built-in importance
importances = best_rf.named_steps["classifier"].feature_importances_ if hasattr(best_rf, "named_steps") else best_rf.feature_importances_

In [10]:
import pandas as pd
from sklearn.inspection import permutation_importance

# Get the column names after preprocessing (so we know what each number represents)
feature_names = preprocessor.get_feature_names_out()

# Method 1: built-in importance
importances = best_rf.feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print("Top 5 features (built-in importance):")
print(importance_df.head(5))

# Method 2: permutation importance (on the held-out test set)
perm_result = permutation_importance(
    best_rf, X_test_processed, y_test,
    n_repeats=10, random_state=42, scoring="roc_auc"
)

perm_df = pd.DataFrame({
    "feature": feature_names,
    "perm_importance": perm_result.importances_mean
}).sort_values("perm_importance", ascending=False)

print("\nTop 5 features (permutation importance):")
print(perm_df.head(5))

Top 5 features (built-in importance):
                      feature  importance
14    cat__payment_method_COD    0.165131
0              num__price_inr    0.137925
2   num__customer_tenure_days    0.107483
5   num__delivery_distance_km    0.096352
1           num__discount_pct    0.090310

Top 5 features (permutation importance):
                            feature  perm_importance
14          cat__payment_method_COD         0.068254
16  cat__payment_method_Prepaid_UPI         0.008012
0                    num__price_inr         0.008002
4         num__num_previous_returns         0.007104
13       cat__product_category_Home         0.003559


**Comparing built-in feature importance to permutation importance reveals that payment_method_COD is genuinely important under both measures — it remains the top feature either way. However, delivery_distance_km, which ranked 4th under built-in importance, drops out of the top 5 entirely under permutation importance, meaning the model's performance barely changes when this column is scrambled — it was never actually relying on it. This happens because impurity-based (built-in) importance can overrate continuous columns with many unique values, since they offer more possible split points to appear useful by chance, even without carrying real predictive signal.**

In [11]:
from sklearn.metrics import recall_score, precision_score

# Get Random Forest's predictions on the test set (using its own predict, default 0.5 threshold)
rf_preds = best_rf.predict(X_test_processed)

# Build a small dataframe to make grouping easy
eval_df = X_test.copy()
eval_df["actual"] = y_test.values
eval_df["predicted"] = rf_preds

print("By product_category:")
for cat in eval_df["product_category"].unique():
    subset = eval_df[eval_df["product_category"] == cat]
    r = recall_score(subset["actual"], subset["predicted"], pos_label=1, zero_division=0)
    p = precision_score(subset["actual"], subset["predicted"], pos_label=1, zero_division=0)
    print(f"{cat}: recall={r:.3f}, precision={p:.3f}, n={len(subset)}")

print("\nBy payment_method:")
for pm in eval_df["payment_method"].unique():
    subset = eval_df[eval_df["payment_method"] == pm]
    r = recall_score(subset["actual"], subset["predicted"], pos_label=1, zero_division=0)
    p = precision_score(subset["actual"], subset["predicted"], pos_label=1, zero_division=0)
    print(f"{pm}: recall={r:.3f}, precision={p:.3f}, n={len(subset)}")

By product_category:
Home: recall=0.676, precision=0.232, n=221
Electronics: recall=0.308, precision=0.267, n=261
Footwear: recall=0.500, precision=0.333, n=217
Apparel: recall=0.530, precision=0.342, n=385
Beauty: recall=0.613, precision=0.487, n=116

By payment_method:
COD: recall=0.877, precision=0.316, n=503
Prepaid_Card: recall=0.000, precision=0.000, n=283
Prepaid_UPI: recall=0.042, precision=0.667, n=294
Wallet: recall=0.048, precision=0.500, n=120


**Breaking down performance by payment method reveals a major weak subgroup: for non-COD payment methods (Prepaid_Card, Prepaid_UPI, Wallet), recall collapses to near 0% (0.0%, 4.2%, and 4.8% respectively), compared to 87.7% recall for COD orders. This suggests the model is over-relying on payment method as a shortcut for risk, since COD is by far its most important feature, leaving it nearly blind to genuine risk signals in non-COD orders. A concrete fix: compute a separate decision threshold per payment method (using the same F1-maximizing threshold sweep from Task 5) rather than applying one global 0.5 cutoff, since non-COD orders' predicted probabilities are clearly compressed into a much lower range than COD orders'**

In [12]:
import joblib
import os

# Step 1: Re-run the threshold sweep, this time on the Random Forest's own probabilities
rf_test_probs = best_rf.predict_proba(X_test_processed)[:, 1]

thresholds = np.arange(0.1, 0.91, 0.02)
rf_results = []

for t in thresholds:
    preds_t = (rf_test_probs >= t).astype(int)
    f1 = f1_score(y_test, preds_t, pos_label=1, zero_division=0)
    recall = recall_score(y_test, preds_t, pos_label=1, zero_division=0)
    precision = precision_score(y_test, preds_t, pos_label=1, zero_division=0)
    rf_results.append((t, f1, recall, precision))

rf_results_df = pd.DataFrame(rf_results, columns=["threshold", "f1", "recall", "precision"])
best_rf_row = rf_results_df.loc[rf_results_df["f1"].idxmax()]

t_star_rf = best_rf_row["threshold"]
print("t*_rf (best threshold for Random Forest):", t_star_rf)
print("F1 at t*_rf:", best_rf_row["f1"])
print("Recall at t*_rf:", best_rf_row["recall"])
print("Precision at t*_rf:", best_rf_row["precision"])

# Step 2: Save the model to a file
os.makedirs("models", exist_ok=True)
joblib.dump(best_rf, "models/return_risk_model.pkl")
print("\nModel saved to models/return_risk_model.pkl")

t*_rf (best threshold for Random Forest): 0.4600000000000001
F1 at t*_rf: 0.39904420549581837
Recall at t*_rf: 0.6117216117216118
Precision at t*_rf: 0.29609929078014185

Model saved to models/return_risk_model.pkl


In [13]:
from sklearn.pipeline import Pipeline

# Combine your already-fitted preprocessor and already-fitted Random Forest into one pipeline
final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", best_rf)
])

# Save this combined pipeline instead
joblib.dump(final_pipeline, "models/return_risk_model.pkl")
print("Combined pipeline saved to models/return_risk_model.pkl")

Combined pipeline saved to models/return_risk_model.pkl


In [14]:
# Test: load it fresh and predict on a raw (not preprocessed) row
loaded_pipeline = joblib.load("models/return_risk_model.pkl")
sample_raw_order = X_test.iloc[[0]]  # a raw, unprocessed row
print(loaded_pipeline.predict_proba(sample_raw_order))

[[0.46796442 0.53203558]]
